# 08 — Repair plans: suggest, approve, apply, undo

Phase 2 makes cleaning **reviewable**: `fd.suggest_plan` proposes exactly the
repairs `fd.clean` would make, you approve/reject/override them, and
`fd.apply_plan` executes *only* what you approved — under a physical
protected-column guard, with drift refusal, an undo log, and a deterministic
audit hash. Everything is offline and model-free.

In [ ]:
import pandas as pd
import freshdata as fd

df = pd.DataFrame({
    "cust_id": ["C001", "C002", "C003", "C004"],
    "email_addr": ["a@@b.com", "x @ y.com", "ok@ok.com", "junk"],
    "mobile": ["98765 43210", "09876543210", "+919876543210", "12345"],
    "monthly_revenue": ["1000", "2000", "3000", "4000"],
    "status": [" Active ", "INACTIVE", "actve", "pending"],
})

CONTEXT = """CustomerID is unique.
Emails must be valid.
Phone numbers are Indian.
Allowed status values are active, inactive, pending.
Never modify revenue values."""

## 1. Suggest a plan

With `semantic_mode="auto"`, low-risk high-confidence actions arrive
pre-approved (they are what `fd.clean` would have applied automatically);
everything ambiguous stays `pending` or is `blocked`.

In [ ]:
plan = fd.suggest_plan(df, context=CONTEXT, semantic_mode="auto", verbose=False)
rp = plan.repair_plan
print(rp.summary())

In [ ]:
rp.to_frame()  # one row per planned action, for filtering/sorting

## 2. Review

Selectors accept an action id, a column name, an action kind, a list of
those, or a predicate.

In [ ]:
# The fuzzy typo "actve" -> "active" was only *suggested* (conf 0.8 < 0.95).
typo = next(a for a in rp.actions if a.params.get("raw_value") == "actve")
rp.approve(typo.id)

# Reject all phone repairs with a recorded reason.
rp.reject("phone_format", reason="numbers are re-synced from the CRM nightly")

# Or approve everything low-risk in one call (never touches blocked/rejected):
rp.approve_all(max_risk="low")
print(rp.summary())

## 3. Apply — exactly what was approved

- nothing is re-profiled, nothing is re-decided;
- rejected and blocked actions do not run;
- the protected-column guard verifies `monthly_revenue` byte-identical;
- the input frame is never mutated.

In [ ]:
clean_df, report = fd.apply_plan(df, rp, keep_undo=True)
clean_df

In [ ]:
print("decisions_hash:", report.decisions_hash)
# Stable for the same reviewed plan; changes when any approval changes.

## 4. Drift refusal

A plan remembers the frame it was built for. Applying it to changed data
refuses by default.

In [ ]:
drifted = df.copy()
drifted.loc[0, "status"] = "archived"
try:
    fd.apply_plan(drifted, rp)
except fd.PlanDriftError as exc:
    print("refused:", exc)

# Explicit override: stale actions are skipped and recorded.
_, drift_report = fd.apply_plan(drifted, rp, allow_drift=True)

## 5. Undo

`keep_undo=True` stores a compact log (cell positions + the raw value per
action), capped by `undo_cell_limit`; actions that don't fit are honestly
marked `reversible=False`.

In [ ]:
email_ids = [a.id for a in rp.actions
             if a.kind == "email_format" and a.approval == "approved"]
restored = report.revert(clean_df, action_ids=email_ids)
restored[["email_addr"]]

## 6. Plans as files

```bash
freshdata plan in.csv --context-file rules.txt --out plan.json
freshdata apply-plan in.csv --plan plan.json -o out.csv --report audit.json
```

`RepairPlan.to_json()` / `from_json()` round-trip actions, approval state,
the compiled policy, and the frame signature — so a plan can be reviewed in
a pull request like code.

## Limitations

- No model, no embeddings, no network; only the deterministic Phase-1
  context language.
- Ambiguous repairs (`junk`, `12345`, typos with two close candidates) are
  suggested or flagged, never auto-applied.
- Undo is cell-scoped; row drops/aggregations are not reversible.